In [6]:
import pandas as pd
import re
from pylatexenc.latexwalker import LatexWalker

In [10]:
def extract_latex_segments(text):
    if not isinstance(text, str):
        return []
    return re.findall(r"\$(.+?)\$", text)

def check_dollar_balance(text):
    if not isinstance(text, str):
        return True
    return text.count("$") % 2 == 0

def latex_valid(expr):
    try:
        walker = LatexWalker(expr)
        nodes, pos, length = walker.get_latex_nodes(pos=0)
        return pos + length == len(expr)
    except Exception:
        return False

def segments_in_reference(segments, reference):
    ref_segments = extract_latex_segments(reference)
    return all(seg in ref_segments for seg in segments)

def check_file(filename, text_col, ref_col):
    
    results = {
        "unbalanced_dollar": [],
        "latex_problem": []
    }

    sheets = pd.read_excel(filename, sheet_name=None)

    for sheet_name, df in sheets.items():
        
        for idx, row in df.iterrows():
            
            text = row[text_col]
            ref = row[ref_col]

            # 1 check $ balance
            if not check_dollar_balance(text):
                results["unbalanced_dollar"].append((sheet_name, idx+2))
                continue

            segments = extract_latex_segments(text)

            # 2 check syntax
            for seg in segments:
                if not latex_valid(seg):
                    results["latex_problem"].append((sheet_name, idx+2))
                    break

            # 3 check match with reference
            if not segments_in_reference(segments, ref):
                results["latex_problem"].append((sheet_name, idx+2))

    return results

In [12]:
problems = check_file(
    "math_translated_copy/polymath_cs.xlsx",
    text_col="questions_translated",
    ref_col="question"
)

print("Unbalanced $:", problems["unbalanced_dollar"])
print("Latex issues:", problems["latex_problem"])

Unbalanced $: [('top', 2), ('top', 6), ('top', 13), ('top', 30), ('top', 36), ('top', 70), ('top', 77), ('top', 106), ('high', 26), ('high', 28), ('high', 48), ('high', 61), ('high', 96), ('high', 103), ('high', 110), ('medium', 26), ('medium', 70), ('medium', 72), ('medium', 119), ('medium', 123), ('medium', 124)]
Latex issues: [('top', 3), ('top', 5), ('top', 8), ('top', 12), ('top', 15), ('top', 17), ('top', 19), ('top', 21), ('top', 22), ('top', 26), ('top', 28), ('top', 29), ('top', 32), ('top', 33), ('top', 34), ('top', 37), ('top', 38), ('top', 44), ('top', 48), ('top', 53), ('top', 59), ('top', 65), ('top', 66), ('top', 67), ('top', 71), ('top', 72), ('top', 73), ('top', 74), ('top', 75), ('top', 79), ('top', 81), ('top', 83), ('top', 86), ('top', 89), ('top', 91), ('top', 92), ('top', 95), ('top', 99), ('top', 103), ('top', 104), ('top', 112), ('top', 113), ('top', 117), ('top', 118), ('top', 119), ('top', 120), ('top', 122), ('top', 126), ('high', 3), ('high', 14), ('high', 1